In [4]:
import pandas as pd
import sqlite3

customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

conn = sqlite3.connect(":memory:")
customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")


# Task 1 — Aggregation and Grouping

# Using only the orders table, write a SQL query that returns each CustomerID along with:
# The total number of orders they placed (order_count)
# The total freight amount across all their orders (total_freight)
# The average freight amount per order (avg_freight)
# Sort the results by total_freight in descending order.

sql_query = """
SELECT
    CustomerID,
    COUNT(OrderID) AS order_count,
    SUM(Freight) AS total_freight,
    AVG(Freight) AS avg_freight
FROM
    orders
GROUP BY
    CustomerID
ORDER BY
    total_freight DESC;
"""

# Run the query using pd.read_sql_query() and display the top 10 rows.
result_df = pd.read_sql_query(sql_query, conn)
print(result_df.head(10))

  customerID  order_count  total_freight  avg_freight
0      SAVEA           31        6683.70   215.603226
1      ERNSH           30        6205.39   206.846333
2      QUICK           28        5605.63   200.201071
3      HUNGO           19        2755.24   145.012632
4      RATTC           18        2134.21   118.567222
5      QUEEN           13        1982.70   152.515385
6      FOLKO           19        1678.08    88.320000
7      BERGS           18        1559.52    86.640000
8      FRANK           15        1403.44    93.562667
9      MEREP           13        1394.22   107.247692


In [8]:
# Task 2 — WHERE vs. HAVING

# Write two separate SQL queries to demonstrate the difference between WHERE and HAVING:
## WE CAN DIRECTELY USE WHERE TO FILTER THE DATA BEFORE AGGREGATION
## WE CAN USE HAVING TO FILTER THE DATA AFTER AGGREGATION
# Query A: From the orders table, filter rows where Freight is greater than 50 before aggregation, then group by CustomerID and return the count of such orders as high_freight_orders.

# Query B: From the orders table, group by CustomerID and return only those customers whose total freight exceeds 500, using HAVING. Return CustomerID and total_freight.

# In a markdown cell below your queries, write 2–3 sentences explaining why Query A and Query B produce different results even though both involve a threshold on Freight.

sql_query_2 = """
SELECT
    CustomerID,
    SUM(Freight) AS total_freight
FROM
    orders
    WHERE Freight > 50
GROUP BY
    CustomerID
    HAVING SUM(Freight) > 500
ORDER BY
    total_freight DESC;
"""
# Run the query using pd.read_sql_query() and display the top 10 rows.
result_df = pd.read_sql_query(sql_query_2, conn)
print(result_df)

   customerID  total_freight
0       SAVEA        6534.44
1       ERNSH        6120.20
2       QUICK        5288.13
3       HUNGO        2518.44
4       RATTC        1950.53
5       QUEEN        1923.68
6       FOLKO        1527.87
7       BERGS        1471.84
8       MEREP        1295.05
9       FRANK        1279.23
10      BONAP        1224.77
11      WHITC        1212.22
12      PICCO        1089.19
13      HILAA        1086.01
14      GREAL         969.59
15      RICSU         950.65
16      VAFFE         876.31
17      OLDWO         865.83
18      SEVES         825.21
19      OTTIK         814.45
20      LEHMS         797.24
21      EASTC         769.16
22      SUPRD         747.64
23      WARTH         677.53
24      HANAR         641.00
25      KOENE         619.59
26      FOLIG         587.98
27      LILAS         573.32
28      BOTTM         557.08
29      BLONP         544.48


Query A and Query B produce different results because of the fundamental difference between WHERE and HAVING clauses. WHERE filters individual rows before any grouping or aggregation takes place. In Query A, only orders with Freight > 50 are considered for counting.

HAVING, on the other hand, filters groups of rows after they have been aggregated. In Query B, all orders for a CustomerID are summed first (SUM(Freight)), and then only those CustomerID groups whose total sum of freight exceeds 500 are included in the final result. The WHERE clause in Query B further refines this by excluding individual orders with Freight <= 50 before the summation for each customer, influencing the SUM(Freight) value used in the HAVING clause.

In [10]:
# Task 3 — JOIN and Aggregation

# Write a SQL query that joins the customers and orders tables on CustomerID and returns:

# CompanyName (from customers)
# Country (from customers)
# Total number of orders placed (order_count)
# Total freight (total_freight)
# Include only customers who have placed at least one order (i.e., use INNER JOIN).

# Then write a second query using LEFT JOIN that includes all customers, even those with no orders. For customers with no orders, total_freight should appear as NULL or 0.

# Display both results and in a markdown cell, explain in 2–3 sentences what changed between the two queries and why.
sql_query_3 = """
SELECT
    C.CompanyName,
    C.Country,
    COUNT(O.orderID) AS ORDER_COUNT,
    SUM(O.FREIGHT) AS TOTAL_FREIGHT
FROM
    CUSTOMERS C
INNER JOIN ORDERS O ON C.CUSTOMERID = O.CUSTOMERID
GROUP BY
    C.CustomerID;
"""
# Run the query using pd.read_sql_query() and display the top 10 rows.
result_df = pd.read_sql_query(sql_query_3, conn)
print(result_df)

                           companyName  country  ORDER_COUNT  TOTAL_FREIGHT
0                  Alfreds Futterkiste  Germany            6         225.58
1   Ana Trujillo Emparedados y helados   Mexico            4          97.42
2              Antonio Moreno Taquería   Mexico            7         268.52
3                      Around the Horn       UK           13         471.95
4                   Berglunds snabbköp   Sweden           18        1559.52
..                                 ...      ...          ...            ...
84                      Wartian Herkku  Finland           15         822.48
85              Wellington Importadora   Brazil            9         194.71
86                White Clover Markets      USA           14        1353.06
87                         Wilman Kala  Finland            7          88.41
88                      Wolski  Zajazd   Poland            7         175.74

[89 rows x 4 columns]


In [11]:
# Task 3 — JOIN and Aggregation

# Write a SQL query that joins the customers and orders tables on CustomerID and returns:

# CompanyName (from customers)
# Country (from customers)
# Total number of orders placed (order_count)
# Total freight (total_freight)
# Include only customers who have placed at least one order (i.e., use INNER JOIN).

# Then write a second query using LEFT JOIN that includes all customers, even those with no orders. For customers with no orders, total_freight should appear as NULL or 0.

# Display both results and in a markdown cell, explain in 2–3 sentences what changed between the two queries and why.
sql_query_3 = """
SELECT
    C.CompanyName,
    C.Country,
    COUNT(O.orderID) AS ORDER_COUNT,
    SUM(O.FREIGHT) AS TOTAL_FREIGHT
FROM
    CUSTOMERS C
LEFT JOIN ORDERS O ON C.CUSTOMERID = O.CUSTOMERID
GROUP BY
    C.CustomerID;
"""
# Run the query using pd.read_sql_query() and display the top 10 rows.
result_df = pd.read_sql_query(sql_query_3, conn)
print(result_df)

                           companyName  country  ORDER_COUNT  TOTAL_FREIGHT
0                  Alfreds Futterkiste  Germany            6         225.58
1   Ana Trujillo Emparedados y helados   Mexico            4          97.42
2              Antonio Moreno Taquería   Mexico            7         268.52
3                      Around the Horn       UK           13         471.95
4                   Berglunds snabbköp   Sweden           18        1559.52
..                                 ...      ...          ...            ...
86                      Wartian Herkku  Finland           15         822.48
87              Wellington Importadora   Brazil            9         194.71
88                White Clover Markets      USA           14        1353.06
89                         Wilman Kala  Finland            7          88.41
90                      Wolski  Zajazd   Poland            7         175.74

[91 rows x 4 columns]


The first query using INNER JOIN returns 89 rows, including only customers who have placed at least one order. This is because INNER JOIN only includes rows where there is a match in *both* the customers and orders tables.

The second query using LEFT JOIN returns 91 rows. This includes all customers from the customers table the 'left' table, even those who have not placed any orders. For customers without orders, the ORDER_COUNT and TOTAL_FREIGHT columns appear as NULL or 0, depending on the aggregation function, as there are no matching entries in the orders table.